In [ ]:
# 1. SETUP - safe to re-run any time. Re-clones the CODE only; never touches result files.
%cd /kaggle/working
!rm -rf StatTrap_SLM
!git clone https://github.com/mirzayasirabdullahbaig07/StatTrap_SLM.git
%cd /kaggle/working/StatTrap_SLM
!git log -1 --oneline
!echo "fix present (must print 1):" && grep -c 'tokenize=False' src/stattrap/backends.py
!pip install -q -e . accelerate bitsandbytes
!stattrap audit-items --per-family 100 --perms 1 --out /tmp/a.csv | tail -3

In [ ]:
# 2. SMOKE TEST (2-3 min, 0.5B model). Look at the table BEFORE running cell 3.
%cd /kaggle/working/StatTrap_SLM
!rm -f /kaggle/working/smoke.csv /kaggle/working/smoke.manifest.json
!stattrap run --backend hf --models Qwen/Qwen2.5-0.5B-Instruct --per-family 5 --perms 2 --limit 33 --max-new-tokens 1024 --out /kaggle/working/smoke.csv
import pandas as pd
pd.set_option('display.max_colwidth', 300)
d = pd.read_csv('/kaggle/working/smoke.csv', keep_default_na=False)
print(d.groupby('style').agg(n=('correct','size'), unparsed=('chosen', lambda s: (s=='').mean()), trunc=('truncated','mean'), acc=('correct','mean')))
for r in d.response.head(4): print('---', r[:300])

In [ ]:
# 3. MAIN RUN, pass 1 (resumable: if the session stops, run this SAME cell again; same --out file)
%cd /kaggle/working/StatTrap_SLM
!stattrap run --backend hf --per-family 30 --perms 2 --max-new-tokens 1024 --out /kaggle/working/main.csv --models Qwen/Qwen2.5-0.5B-Instruct Qwen/Qwen2.5-1.5B-Instruct Qwen/Qwen2.5-3B-Instruct HuggingFaceTB/SmolLM2-1.7B-Instruct

In [ ]:
# 4. The 7B model in 4-bit (long: possibly 3-4 h). Needs the SAME --per-family/--perms and SAME --out as cell 3.
%cd /kaggle/working/StatTrap_SLM
!stattrap run --backend hf --load-in-4bit --per-family 30 --perms 2 --max-new-tokens 1024 --out /kaggle/working/main.csv --models Qwen/Qwen2.5-7B-Instruct

In [ ]:
# 5. ANALYSIS - tables and figures are written next to main.csv in /kaggle/working
%cd /kaggle/working/StatTrap_SLM
!stattrap analyze --csv /kaggle/working/main.csv
!ls /kaggle/working

In [ ]:
# 6. ZIP the results, then download stattrap_results.zip from the Output panel BEFORE the session ends
!cd /kaggle/working && zip -rq stattrap_results.zip main.csv main.manifest.json main_figures main_*.csv && ls -la stattrap_results.zip